<a href="https://colab.research.google.com/github/R4Z47/nba-win-predictor/blob/main/Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Predicting Sports Winners
Data Science and Applied Machine Learning <br>
Adpopted from material from Dr. Kerby <br>
Due 4/15/26 by 11:59 PM <br>
100 Points <br>
Student Name: Razat Sangraula

### Question 1: Gather Data from 2024-2025
*Gather* data from [Basketball Reference](https://www.basketball-reference.com/leagues/NBA_2025.html) <br>

Data can be downloaded by clicking on the `Share and Export` drop-down list at the middle top of available tables. Start off by clicking `Schedule and Results` and downloading the monthly games tables. Combine these into one table. Then import the full games-results table into python (make sure to include your dataset with your submission).


In [11]:
import pandas as pd

oct = pd.read_csv('oct.csv')
nov = pd.read_csv('nov.csv')
dec = pd.read_csv('dec.csv')
jan = pd.read_csv('jan.csv')
feb = pd.read_csv('feb.csv')
mar = pd.read_csv('mar.csv')
apr = pd.read_csv('apr.csv')

def clean(df):
    df = df[df['PTS'].apply(lambda x: str(x).strip().isdigit())]
    df['PTS']   = df['PTS'].astype(int)
    df['PTS.1'] = df['PTS.1'].astype(int)
    return df

oct = clean(oct)
nov = clean(nov)
dec = clean(dec)
jan = clean(jan)
feb = clean(feb)
mar = clean(mar)
apr = clean(apr)

sports = pd.concat([oct, nov, dec, jan, feb, mar, apr], ignore_index=True)

print(f"Total games : {len(sports)}")
print(f"Date range  : {sports['Date'].iloc[0]}  →  {sports['Date'].iloc[-1]}")
print(f"Shape       : {sports.shape}")
sports

Total games : 1275
Date range  : Tue Oct 22 2024  →  Wed Apr 30 2025
Shape       : (1275, 12)


,Date,Start (ET),Visitor/Neutral,PTS,Home/Neutral,PTS.1,Unnamed: 6,Unnamed: 7,Attend.,LOG,Arena,Notes
0,Tue Oct 22 2024,7:30p,New York Knicks,109,Boston Celtics,132,Box Score,NaN,19156,2:04,TD Garden,NaN
1,Tue Oct 22 2024,10:00p,Minnesota Timberwolves,103,Los Angeles Lakers,110,Box Score,NaN,18997,2:26,Crypto.com Arena,NaN
2,Wed Oct 23 2024,7:00p,Indiana Pacers,115,Detroit Pistons,109,Box Score,NaN,20062,2:23,Little Caesars Arena,NaN
3,Wed Oct 23 2024,7:30p,Brooklyn Nets,116,Atlanta Hawks,120,Box Score,NaN,17548,2:33,State Farm Arena,NaN
4,Wed Oct 23 2024,7:30p,Orlando Magic,116,Miami Heat,97,Box Score,NaN,19630,2:31,Kaseya Center,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1270,Tue Apr 29 2025,7:30p,Detroit Pistons,106,New York Knicks,103,Box Score,NaN,19812,2:46,Madison Square Garden (IV),NaN
1271,Tue Apr 29 2025,8:30p,Orlando Magic,89,Boston Celtics,120,Box Score,NaN,19156,2:19,TD Garden,NaN
1272,Tue Apr 29 2025,10:00p,Los Angeles Clippers,115,Denver Nuggets,131,Box Score,NaN,20005,2:29,Ball Arena,NaN
1273,Wed Apr 30 2025,7:30p,Golden State Warriors,116,Houston Rockets,131,Box Score,NaN,18055,2:31,Toyota Center,NaN


### Question 2: EDA and Preprocessing
Explore your dataset and make any preprocessing adjustments necessary. You must perform at least 3 operations on exploring your data. Each one must provide unqiue insight to the data. At the bottom of this problem, write a paragraph (at least 5 sentences) explaining what you did and found out.

In [12]:
sports.drop(columns=['Start (ET)', 'Unnamed: 6', 'Unnamed: 7', 'LOG', 'Arena', 'Notes'], inplace=True)
sports.rename(columns={'Visitor/Neutral':'Visitor Team','Home/Neutral':'Home Team','PTS':'PTS_Vis','PTS.1':'PTS_Home'}, inplace=True)

print(sports.isnull().sum())
print(sports[['PTS_Vis', 'PTS_Home']].describe())
print((sports['PTS_Home'] > sports['PTS_Vis']).mean())
print(sports['Attend.'].mean())
print(sports['Home Team'].value_counts())

sports.drop(columns=['Attend.'], inplace=True)
sports.head()

Date            0
Visitor Team    0
PTS_Vis         0
Home Team       0
PTS_Home        0
Attend.         0
dtype: int64
           PTS_Vis     PTS_Home
count  1275.000000  1275.000000
mean    112.750588   114.426667
std      12.877965    12.628675
min      67.000000    79.000000
25%     104.000000   105.000000
50%     113.000000   115.000000
75%     121.000000   123.000000
max     162.000000   155.000000
0.5450980392156862
18167.616470588237
Home Team
Oklahoma City Thunder     45
Boston Celtics            44
Houston Rockets           44
Milwaukee Bucks           44
Denver Nuggets            44
Los Angeles Lakers        44
New York Knicks           44
Indiana Pacers            44
Golden State Warriors     44
Memphis Grizzlies         44
Orlando Magic             44
Detroit Pistons           43
Los Angeles Clippers      43
Miami Heat                43
Cleveland Cavaliers       43
Minnesota Timberwolves    43
Chicago Bulls             42
Sacramento Kings          42
Toronto Raptors      

,Date,Visitor Team,PTS_Vis,Home Team,PTS_Home
0,Tue Oct 22 2024,New York Knicks,109,Boston Celtics,132
1,Tue Oct 22 2024,Minnesota Timberwolves,103,Los Angeles Lakers,110
2,Wed Oct 23 2024,Indiana Pacers,115,Detroit Pistons,109
3,Wed Oct 23 2024,Brooklyn Nets,116,Atlanta Hawks,120
4,Wed Oct 23 2024,Orlando Magic,116,Miami Heat,97


The dataset has 1,275 NBA games from the 2024-25 regular season with all 30 teams represented. The first thing I checked was whether there were any missing values. There weren't, which made things easier. Looking at the scores, home teams averaged about 114 points per game while visitors averaged around 113, a small but consistent gap. Home teams ended up winning about 54.5% of games, which lines up with what most people already know about home court advantage in the NBA. I'm also noticed that every team played between 40 and 45 home games, which confirms the schedule was balanced across the season.

### Question 3: Feature Engineering

Create a column for whether or not the home team won the game; call it `HomeWin`. Use Pandas and your knowledge of Python to fill in this data. This will become our target -- ie what we are trying to predict.

In [13]:
sports['HomeWin'] = (sports['PTS_Home'] > sports['PTS_Vis']).astype(int)
sports.head()

,Date,Visitor Team,PTS_Vis,Home Team,PTS_Home,HomeWin
0,Tue Oct 22 2024,New York Knicks,109,Boston Celtics,132,1
1,Tue Oct 22 2024,Minnesota Timberwolves,103,Los Angeles Lakers,110,1
2,Wed Oct 23 2024,Indiana Pacers,115,Detroit Pistons,109,0
3,Wed Oct 23 2024,Brooklyn Nets,116,Atlanta Hawks,120,1
4,Wed Oct 23 2024,Orlando Magic,116,Miami Heat,97,0


### Question 4: Feature Engineering II
Create two columns for how many games the home and visitor teams have won thus far in the season. Call them `HomeNumWins` and `VisitorNumWins`.

For example Game 1 for the Utah Jazz was against the Menphis Grizzlies. The Jazz were the home team, Memphis was the road team. Since this is game 1 for both teams `HomeNumWins` and `VisitorNumWins` will be 0. The Jazz lost this game and were once again the home team for their next game (against the Golden State Warriors),  `HomeNumWins` should be a 0 for this value. Memphis would go on to lose their second game (on the road to Houston), Game 3 was their first home game. So for OKC's 3rd game (against the Magic) `HomeNumWins` would be 1.

Hint: Store each team and the number of intial wins (0) in a Python dictionary. You can then increment this value as you go through the dataset and use this information to update your feature.



In [14]:
from collections import defaultdict

won_last = defaultdict(int)
num_wins = defaultdict(int)

sports['HomeLastWin'] = 0
sports['VisitorLastWin'] = 0
sports['HomeNumWins'] = 0
sports['VisitorNumWins'] = 0

for index, row in sports.iterrows():
    home_team = row['Home Team']
    visitor_team = row['Visitor Team']

    sports.loc[index, 'HomeLastWin'] = won_last[home_team]
    sports.loc[index, 'VisitorLastWin'] = won_last[visitor_team]
    sports.loc[index, 'HomeNumWins'] = num_wins[home_team]
    sports.loc[index, 'VisitorNumWins'] = num_wins[visitor_team]

    won_last[home_team] = int(row['HomeWin'])
    won_last[visitor_team] = 1 - int(row['HomeWin'])
    num_wins[home_team] += int(row['HomeWin'])
    num_wins[visitor_team] += 1 - int(row['HomeWin'])

sports.head(20)

,Date,Visitor Team,PTS_Vis,Home Team,PTS_Home,HomeWin,HomeLastWin,VisitorLastWin,HomeNumWins,VisitorNumWins
0,Tue Oct 22 2024,New York Knicks,109,Boston Celtics,132,1,0,0,0,0
1,Tue Oct 22 2024,Minnesota Timberwolves,103,Los Angeles Lakers,110,1,0,0,0,0
2,Wed Oct 23 2024,Indiana Pacers,115,Detroit Pistons,109,0,0,0,0,0
3,Wed Oct 23 2024,Brooklyn Nets,116,Atlanta Hawks,120,1,0,0,0,0
4,Wed Oct 23 2024,Orlando Magic,116,Miami Heat,97,0,0,0,0,0
5,Wed Oct 23 2024,Milwaukee Bucks,124,Philadelphia 76ers,109,0,0,0,0,0
6,Wed Oct 23 2024,Cleveland Cavaliers,136,Toronto Raptors,106,0,0,0,0,0
7,Wed Oct 23 2024,Charlotte Hornets,110,Houston Rockets,105,0,0,0,0,0
8,Wed Oct 23 2024,Chicago Bulls,111,New Orleans Pelicans,123,1,0,0,0,0
9,Wed Oct 23 2024,Memphis Grizzlies,126,Utah Jazz,124,0,0,0,0,0


### Question 5: Model

Train a decision tree model to determine if the home team won (a 1 is a win and a 0 is a loss. Evaluate your model. At the bottom of this problem write 2 paragraphs explaining your models, its performance and any possible issues you see with it. Comment on the most "important" features in the model.

Remember to split the dataset into training and testing sets and delete any columns that you deem unuseful. (Hint: Do not use any information that you wouldn't know before the game is played, the idea is to create a model that predicts wins)



In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = sports[['HomeLastWin', 'VisitorLastWin', 'HomeNumWins', 'VisitorNumWins']]
y = sports['HomeWin']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

print(f"Training accuracy: {accuracy_score(y_train, dt.predict(X_train)):.3f}")
print(f"Testing accuracy : {accuracy_score(y_test, dt.predict(X_test)):.3f}")
print(f"Baseline         : {y.mean():.3f}")
print()
print(classification_report(y_test, dt.predict(X_test), target_names=['Away Win', 'Home Win']))
print("Feature importances:")
for f, i in zip(X.columns, dt.feature_importances_):
    print(f"  {f}: {i:.4f}")

Training accuracy: 0.695
Testing accuracy : 0.592
Baseline         : 0.545

              precision    recall  f1-score   support

    Away Win       0.62      0.46      0.53       127
    Home Win       0.57      0.73      0.64       128

    accuracy                           0.59       255
   macro avg       0.60      0.59      0.58       255
weighted avg       0.60      0.59      0.58       255

Feature importances:
  HomeLastWin: 0.0108
  VisitorLastWin: 0.0259
  HomeNumWins: 0.4346
  VisitorNumWins: 0.5287


I trained a Decision Tree with a max depth of 5 using only features I'd know before the game tips off whether each team won their last game and how many wins they have so far this season. I split the data 80/20 for training and testing. The model hit 69.5% on training and 59.2% on testing, which isn't amazing but it's still better than just picking the home team every time, which would only get you 54.5%.


Looking at the feature importances, VisitorNumWins (0.53) and HomeNumWins (0.43) did basically all the work how good each team is overall matters way more than whether they won their last game. HomeLastWin and VisitorLastWin were almost irrelevant. The biggest issue I see is the gap between training and testing accuracy, which tells me the model is probably overfitting a bit. It's also pretty limited since it has no idea about things like injuries, back-to-back games, or travel, which probably matter a lot in real life.

### Question 6: Add data from the previous season
Go back to the website and download the team standings from the PREVIOUS season. You will find it in the `Standings` tab. Keep just the number of wins and create two new columns called `HomeWins2024` and `VisitorWins2024`.

In [16]:
prev_wins = {
    "Boston Celtics": 64,
    "Denver Nuggets": 57,
    "Oklahoma City Thunder": 57,
    "Minnesota Timberwolves": 56,
    "Los Angeles Clippers": 51,
    "Dallas Mavericks": 50,
    "New York Knicks": 50,
    "Milwaukee Bucks": 49,
    "New Orleans Pelicans": 49,
    "Phoenix Suns": 49,
    "Cleveland Cavaliers": 48,
    "Indiana Pacers": 47,
    "Los Angeles Lakers": 47,
    "Orlando Magic": 47,
    "Philadelphia 76ers": 47,
    "Miami Heat": 46,
    "Golden State Warriors": 46,
    "Sacramento Kings": 46,
    "Houston Rockets": 41,
    "Chicago Bulls": 39,
    "Atlanta Hawks": 36,
    "Brooklyn Nets": 32,
    "Utah Jazz": 31,
    "Memphis Grizzlies": 27,
    "Toronto Raptors": 25,
    "Charlotte Hornets": 21,
    "Portland Trail Blazers": 21,
    "San Antonio Spurs": 22,
    "Washington Wizards": 15,
    "Detroit Pistons": 14,
}

sports['HomeWins2024'] = sports['Home Team'].map(prev_wins)
sports['VisitorWins2024'] = sports['Visitor Team'].map(prev_wins)

sports.head()

,Date,Visitor Team,PTS_Vis,Home Team,PTS_Home,HomeWin,HomeLastWin,VisitorLastWin,HomeNumWins,VisitorNumWins,HomeWins2024,VisitorWins2024
0,Tue Oct 22 2024,New York Knicks,109,Boston Celtics,132,1,0,0,0,0,64,50
1,Tue Oct 22 2024,Minnesota Timberwolves,103,Los Angeles Lakers,110,1,0,0,0,0,47,56
2,Wed Oct 23 2024,Indiana Pacers,115,Detroit Pistons,109,0,0,0,0,0,14,47
3,Wed Oct 23 2024,Brooklyn Nets,116,Atlanta Hawks,120,1,0,0,0,0,36,32
4,Wed Oct 23 2024,Orlando Magic,116,Miami Heat,97,0,0,0,0,0,46,47


### Question 7: Repeat Model

Repeat your analysis from Question 5.

In [17]:
X = sports[['HomeLastWin', 'VisitorLastWin', 'HomeNumWins', 'VisitorNumWins', 'HomeWins2024', 'VisitorWins2024']]
y = sports['HomeWin']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

print(f"Training accuracy: {accuracy_score(y_train, dt.predict(X_train)):.3f}")
print(f"Testing accuracy : {accuracy_score(y_test, dt.predict(X_test)):.3f}")
print(f"Baseline         : {y.mean():.3f}")
print()
print(classification_report(y_test, dt.predict(X_test), target_names=['Away Win', 'Home Win']))
print("Feature importances:")
for f, i in zip(X.columns, dt.feature_importances_):
    print(f"  {f}: {i:.4f}")

Training accuracy: 0.697
Testing accuracy : 0.600
Baseline         : 0.545

              precision    recall  f1-score   support

    Away Win       0.65      0.43      0.52       127
    Home Win       0.58      0.77      0.66       128

    accuracy                           0.60       255
   macro avg       0.61      0.60      0.59       255
weighted avg       0.61      0.60      0.59       255

Feature importances:
  HomeLastWin: 0.0263
  VisitorLastWin: 0.0000
  HomeNumWins: 0.1470
  VisitorNumWins: 0.2560
  HomeWins2024: 0.2562
  VisitorWins2024: 0.3145


I repeated the same Decision Tree model but this time added HomeWins2024 and VisitorWins2024 as extra features, giving the model some context about how each team performed the previous season. Training accuracy was 69.7% and testing accuracy came in at 60.0%, which is a small improvement over the first model's 59.2%. It's still well above the baseline of 54.5%, so the previous season wins did add a little bit of useful information.


The most important feature this time was VisitorWins2024 (0.31), followed closely by HomeWins2024 (0.26) and VisitorNumWins (0.26). This makes sense how good a team was last season is a decent indicator of how good they are this season. VisitorLastWin had zero importance again, which confirms that winning your last game really doesn't matter much for predicting the next one. The model still has the same overfitting issue with a gap between training and testing accuracy, and it still doesn't know anything about injuries or scheduling.

### Question 8: Analysis and Discussion
What was your best-performing model? \\
How well did it do? \\
Was it overfit? \\
What seems to be the most important feature(s)? \\
How well did your model perform compared to blindly guessing the Home Team won?

My best performing model was the Question 7 model, which included the previous season wins alongside the current season features. It hit 60.0% testing accuracy compared to 59.2% from the first model, so adding the 2023-24 win totals did help a little. The model was somewhat overfit.Training accuracy was 69.7% while testing was only 60.0%, which is about a 10 point gap. That tells me the tree learned some patterns in the training data that didn't fully carry over to new games. The most important features were VisitorWins2024 (0.31), HomeWins2024 (0.26), and VisitorNumWins (0.26), meaning how good a team is overall. Both this season and last matters way more than recent momentum like winning their last game. Compared to just blindly guessing the home team wins every single game, which would give you 54.5% accuracy, my best model at 60.0% is a clear improvement of about 5.5 percentage points. It's not a perfect model by any means, but it shows that team quality is a real predictor of game outcomes even without knowing anything about injuries, rest days, or matchups.